<a href="https://colab.research.google.com/github/AlperYildirim1/Oppenheim-Lim-Neural-Networks/blob/main/Train_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# CELL 1 — IMPORTS & CONFIG
# ============================================================================
# Paste each "CELL" block into its own Colab cell. Sections are independent
# enough to re-run individually after edits.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- Global config (edit here, re-run everything below) ----
CFG = dict(
    img_size      = 224,
    patch_size    = 16,
    in_chans      = 3,
    num_classes   = 100,        # ImageNet-100 -> 100 | ImageNet-1k -> 1000 | CIFAR-100 -> 100
    d_model       = 256,        # ~GFNet-ti width. 384 ~ xs-ish, 512 ~ s-ish
    depth         = 12,
    dropout       = 0.1,
    filter_hidden = 64,         # NeuralFilter2D MLP width
    fft_mode      = "spatial",  # "spatial" = FFT over (H, W) only  -> GFNet-comparable (DEFAULT)
                                # "spatial_channel" = FFT over (H, W, D) -> FNet-style ablation
    activation    = "crelu",  # "modrelu" (constrained / phase-preserving) | "crelu" (unconstrained)
    grad_ckpt     = False,       # gradient checkpointing (saves VRAM on Colab)
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
# NOTE: complex FFT kernels run in FP32 under autocast (no complex BF16/FP16).
# Keep AMP on for the real-valued parts; PyTorch handles the fallback.


# ============================================================================
# CELL 2 — COMPLEX PRIMITIVES
# ============================================================================

class RobustPhaseNorm(nn.Module):
    """RMS norm on magnitudes; phase untouched. Works on [..., D]."""
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        mag = torch.abs(x)
        rms = torch.sqrt(torch.mean(mag ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale


class ModReLU(nn.Module):
    """Rectifies magnitude, strictly preserves phase."""
    def __init__(self, features):
        super().__init__()
        self.b = nn.Parameter(torch.zeros(features))

    def forward(self, z):
        mag = torch.abs(z)
        new_mag = F.relu(mag + self.b)
        phase = z / (mag + 1e-6)
        return new_mag * phase


class CReLU(nn.Module):
    """Independent ReLU on Re/Im. Phase-destroying (quadrant gating)."""
    def __init__(self, features=None):
        super().__init__()

    def forward(self, z):
        return torch.complex(F.relu(z.real), F.relu(z.imag))


class ComplexDropout(nn.Module):
    """One shared Bernoulli mask for Re and Im -> drops amplitude AND phase together."""
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p

    def forward(self, z):
        if not self.training or self.p == 0.0:
            return z
        mask = torch.ones_like(z.real)
        mask = F.dropout(mask, self.p, self.training, inplace=False)
        return z * mask


class ComplexToRealBridge(nn.Module):
    """Per-token C^d -> R^d. The ONLY place a phase-destroying LayerNorm lives
    in the constrained model (we're already leaving the complex domain here)."""
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x_complex):
        cat = torch.cat([x_complex.real, x_complex.imag], dim=-1)
        return self.norm(self.proj(cat))


# ============================================================================
# CELL 3 — DYNAMIC RoSE 2D (patch embed + content wave + 2D static rotation)
# ============================================================================
# Changes vs 1D: nn.Embedding -> Conv2d patch embed; static rotation encodes
# (row, col): first half of dims rotated by ROW angle, second half by COL angle
# (standard 2D-RoPE construction). Adapter + dynamic rotation are per-token
# MLPs -> transfer verbatim. All position tensors are built at RUNTIME grid
# size -> resolution extrapolation works by construction.

class DynamicRoSE2D(nn.Module):
    def __init__(self, d_model, patch_size=16, in_chans=3, max_period=10000.0):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even for the row/col split"
        self.d_model = d_model

        # 1. Patch embed (the "particle")
        self.patch_embed = nn.Conv2d(in_chans, d_model,
                                     kernel_size=patch_size, stride=patch_size)

        # 2. Complex adapter (the "wave" content)
        self.adapter = nn.Linear(d_model, d_model * 2)

        # 3. Dynamic content-dependent rotation (phi_steer)
        self.rotation_predictor = nn.Linear(d_model, d_model * 2)

        # 4. Static 2D positional frequencies (half dims for rows, half for cols)
        half = d_model // 2
        freqs = torch.exp(torch.arange(0, half, dtype=torch.float32)
                          * -(math.log(max_period) / half))
        self.register_buffer("freqs", freqs)  # [D/2]

    def _static_rotation(self, H, W, device):
        rows = torch.arange(H, device=device).float()
        cols = torch.arange(W, device=device).float()
        ang_r = torch.outer(rows, self.freqs)            # [H, D/2]
        ang_c = torch.outer(cols, self.freqs)            # [W, D/2]
        ang_r = ang_r[:, None, :].expand(H, W, -1)       # [H, W, D/2]
        ang_c = ang_c[None, :, :].expand(H, W, -1)       # [H, W, D/2]
        angles = torch.cat([ang_r, ang_c], dim=-1)       # [H, W, D]
        return torch.polar(torch.ones_like(angles), angles)

    def forward(self, images):
        # A. Particle: [B, 3, H_img, W_img] -> [B, H, W, D]
        x = self.patch_embed(images)                     # [B, D, H, W]
        real_base = x.permute(0, 2, 3, 1).contiguous()   # [B, H, W, D]
        B, H, W, D = real_base.shape

        # B. Wave content (fp32: torch.complex does not accept bf16 under autocast)
        cp = self.adapter(real_base).float()
        z_t = torch.complex(cp[..., :D], cp[..., D:])

        # C. Dynamic rotation (unit modulus)
        rr = self.rotation_predictor(real_base).float()
        rx, ry = rr.chunk(2, dim=-1)
        rmag = torch.sqrt(rx ** 2 + ry ** 2 + 1e-6)
        dyn_rot = torch.complex(rx / rmag, ry / rmag)

        # D. Static 2D rotation (built at runtime grid size)
        static_rot = self._static_rotation(H, W, images.device)  # [H, W, D]

        z_final = z_t * static_rot.unsqueeze(0) * dyn_rot
        return z_final, real_base                        # wave [B,H,W,D] cplx, particle real


# ============================================================================
# CELL 4 — NEURAL FILTER 2D (implicit MLP filter over the (row, col) freq grid)
# ============================================================================
# 1D version mapped t -> C^{L x D}. 2D maps (r, c) -> C^{H x W x D}.
# Generated at the INCOMING grid size every forward -> no truncate/pad path,
# no interpolation at new resolutions

class NeuralFilter2D(nn.Module):
    def __init__(self, d_model, hidden_dim=64):
        super().__init__()
        self.d_model = d_model
        nf = hidden_dim // 2
        freqs = torch.exp(torch.arange(0, nf, dtype=torch.float32)
                          * -(math.log(10000.0) / nf))
        self.register_buffer("freqs", freqs)             # [hidden/2]
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, d_model * 2),
        )
        # cache: filters are input-independent per grid size
        self._cache_key, self._cache_val = None, None

    def _embed(self, t):                                  # t: [N, 1] in [0,1]
        ang = t * self.freqs                              # [N, hidden/2]
        return torch.cat([torch.sin(ang), torch.cos(ang)], dim=-1)  # [N, hidden]

    def forward(self, H, W, device):
        key = (H, W, device, self.training)
        if (not self.training) and self._cache_key == key:
            return self._cache_val

        r = torch.linspace(0, 1, steps=H, device=device).unsqueeze(-1)
        c = torch.linspace(0, 1, steps=W, device=device).unsqueeze(-1)
        er = self._embed(r)                               # [H, hidden]
        ec = self._embed(c)                               # [W, hidden]
        grid = torch.cat([
            er[:, None, :].expand(H, W, -1),
            ec[None, :, :].expand(H, W, -1),
        ], dim=-1)                                        # [H, W, 2*hidden]
        out = self.mlp(grid).float().view(H, W, self.d_model, 2)  # fp32 for torch.complex
        h = torch.complex(out[..., 0], out[..., 1])       # [H, W, D]

        if not self.training:
            self._cache_key, self._cache_val = key, h
        return h


# ============================================================================
# CELL 5 — GATED HARMONIC CONVOLUTION 2D (Lean C-GLU block, on a grid)
# ============================================================================
# fft_mode="spatial":          FFT over (H, W) -> channels untouched (GFNet-comparable, DEFAULT)
# fft_mode="spatial_channel":  FFT over (H, W, D) -> FNet-style experimental mixing (ablation)
# activation: ModReLU (constrained) vs CReLU (unconstrained), as in the paper's Table 4.

class GatedHarmonicConvolution2D(nn.Module):
    def __init__(self, d_model, dropout=0.1,
                 fft_mode="spatial", activation="modrelu", filter_hidden=64):
        super().__init__()
        assert fft_mode in ("spatial", "spatial_channel")
        self.d_model = d_model
        self.fft_mode = fft_mode
        self.neural_filter = NeuralFilter2D(d_model, hidden_dim=filter_hidden)

        # ---- Lean C-GLU (verbatim structure from 1D code) ----
        self.value_proj = nn.Linear(d_model * 2, d_model * 2)
        self.gate_proj  = nn.Linear(d_model * 2, d_model * 2)
        self.out_real   = nn.Linear(d_model, d_model)
        self.out_imag   = nn.Linear(d_model, d_model)

        self.activation = ModReLU(d_model) if activation == "modrelu" else CReLU(d_model)
        self.norm = RobustPhaseNorm(d_model)
        self.dropout = ComplexDropout(dropout)

    def forward(self, x):                                 # x: [B, H, W, D] complex
        residual = x
        x_norm = self.norm(x)
        B, H, W, D = x_norm.shape

        # ---- Spectral mixing at the INCOMING grid size (no truncate/pad) ----
        if self.fft_mode == "spatial":
            x_freq = torch.fft.fft2(x_norm, dim=(1, 2), norm="ortho")
            h = self.neural_filter(H, W, x.device).unsqueeze(0)        # [1,H,W,D]
            x_time = torch.fft.ifft2(x_freq * h, dim=(1, 2), norm="ortho")
        else:  # spatial_channel
            x_freq = torch.fft.fftn(x_norm, dim=(1, 2, 3), norm="ortho")
            h = self.neural_filter(H, W, x.device).unsqueeze(0)
            x_time = torch.fft.ifftn(x_freq * h, dim=(1, 2, 3), norm="ortho")

        # ---- Lean C-GLU (per-token; identical to 1D version) ----
        x_cat = torch.cat([x_time.real, x_time.imag], dim=-1)          # [B,H,W,2D]
        v_raw = self.value_proj(x_cat).float()   # fp32: torch.complex rejects bf16
        g_raw = self.gate_proj(x_cat).float()
        Z_value = torch.complex(v_raw[..., :D], v_raw[..., D:])
        Z_gate  = torch.complex(g_raw[..., :D], g_raw[..., D:])
        x_gated = Z_value * Z_gate                                     # pure complex product

        x_act = self.activation(x_gated)

        out = torch.complex(
            (self.out_real(x_act.real) - self.out_imag(x_act.imag)).float(),
            (self.out_real(x_act.imag) + self.out_imag(x_act.real)).float())

        return self.dropout(out) + residual


# ============================================================================
# CELL 6 — PRISM-2D CLASSIFIER (encoder -> phase-preserving norm -> bridge -> GAP -> head)
# ============================================================================
# No attention refiner. Final norm is RobustPhaseNorm (phase-preserving);
# the only LayerNorm sits inside the bridge, after we leave the complex domain.
# forward(images) -> logits, so it drops into GFNet's training harness as an arch.

class PRISM2D(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=100,
                 d_model=256, depth=12, dropout=0.1,
                 fft_mode="spatial", activation="modrelu",
                 filter_hidden=64, grad_ckpt=False):
        super().__init__()
        self.grad_ckpt = grad_ckpt

        self.rose = DynamicRoSE2D(d_model, patch_size, in_chans)
        self.layers = nn.ModuleList([
            GatedHarmonicConvolution2D(d_model, dropout, fft_mode, activation, filter_hidden)
            for _ in range(depth)
        ])
        self.final_norm = RobustPhaseNorm(d_model)        # phase-preserving by construction
        self.bridge = ComplexToRealBridge(d_model)
        self.head = nn.Linear(d_model, num_classes)

        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward_features(self, images):
        x, _particle = self.rose(images)                  # [B, H, W, D] complex
        for layer in self.layers:
            if self.training and self.grad_ckpt:
                x = torch.utils.checkpoint.checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        x = self.final_norm(x)
        x = self.bridge(x)                                # [B, H, W, D] real
        return x.mean(dim=(1, 2))                         # GAP -> [B, D]

    def forward(self, images):
        return self.head(self.forward_features(images))


def build_prism2d(cfg=CFG):
    return PRISM2D(
        img_size=cfg["img_size"], patch_size=cfg["patch_size"], in_chans=cfg["in_chans"],
        num_classes=cfg["num_classes"], d_model=cfg["d_model"], depth=cfg["depth"],
        dropout=cfg["dropout"], fft_mode=cfg["fft_mode"], activation=cfg["activation"],
        filter_hidden=cfg["filter_hidden"], grad_ckpt=cfg["grad_ckpt"],
    )


# ============================================================================
# CELL 7 — SMOKE TEST (shapes, params, gradients, resolution extrapolation)
# ============================================================================
# Run this before anything else. Then run it again with
# CFG["fft_mode"]="spatial_channel" and CFG["activation"]="crelu" to check
# all four variant combinations build and backprop.

def smoke_test():
    model = build_prism2d().to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: {n_params/1e6:.2f}M | fft_mode={CFG['fft_mode']} | act={CFG['activation']}")

    # forward + backward at training resolution
    model.train()
    x = torch.randn(2, 3, CFG["img_size"], CFG["img_size"], device=DEVICE)
    logits = model(x)
    loss = logits.float().logsumexp(-1).mean()
    loss.backward()
    print("224 ->", tuple(logits.shape), "| backward OK")

    # zero-shot resolution extrapolation (the GFNet-can't-do-this-cleanly test)
    model.eval()
    with torch.no_grad():
        for res in (224, 384, 512):
            y = model(torch.randn(2, 3, res, res, device=DEVICE))
            grid = res // CFG["patch_size"]
            print(f"{res} (grid {grid}x{grid}) ->", tuple(y.shape))

smoke_test()

In [ ]:
# ============================================================================
# CELL T1 — TRAINING CONFIG
# ============================================================================
# Prerequisites in the notebook BEFORE these cells:
#   - Drive mounted at /content/drive
#   - !pip install timm datasets
#   - The PRISM2D cells (prism2d_cells.py) already run, so PRISM2D / build_prism2d exist.

import os, math, csv, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
from torch.utils.data import DataLoader

TRAIN_CFG = dict(
    arch          = "prism2d",        # "gfnet_ti" | "prism2d" | "prism2d_crelu" | "prism2d_chanfft"
    run_name      = None,              # default: arch name. Set manually for ablations.
    num_classes   = 100,
    img_size      = 224,
    epochs        = 100,
    warmup_epochs = 5,
    batch_size    = 256,               # A100-40GB friendly at d=256; drop to 128 if OOM
    lr            = 1e-3,              # peak LR for batch 256 (DeiT-ti convention: 5e-4 * bs/512)
    min_lr        = 1e-5,
    weight_decay  = 0.05,              # GFNet/DeiT convention. PRISM-constrained override below.
    label_smooth  = 0.1,
    mixup_alpha   = 0.8,
    cutmix_alpha  = 1.0,
    drop_path     = 0.1,               # GFNet only (stochastic depth)
    seed          = 42,
    num_workers   = 12,
    amp_dtype     = torch.bfloat16,    # A100: bf16, no GradScaler needed.
                                       # NOTE: complex FFT ops auto-fallback to FP32 inside autocast.
    drive_root    = "/content/drive/MyDrive/prism2d_runs",
    data_local    = "/content/data",   # local SSD copy (NEVER train reading from Drive)
)

# Per-arch overrides
ARCH_OVERRIDES = {
    "prism2d":         dict(weight_decay=0.05),
    "prism2d_crelu":   dict(weight_decay=0.05),
    "prism2d_chanfft": dict(weight_decay=0.05),
}

def make_run_cfg(cfg=TRAIN_CFG):
    c = dict(cfg)
    c.update(ARCH_OVERRIDES.get(c["arch"], {}))
    c["run_name"] = c["run_name"] or c["arch"]
    c["run_dir"] = os.path.join(c["drive_root"], c["run_name"])
    os.makedirs(c["run_dir"], exist_ok=True)
    return c

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================================
# CELL T2 — DATA: ImageNet-100 (HuggingFace -> Drive cache -> local SSD)
# ============================================================================
# First session: downloads from HF and saves to Drive (~16GB, one-time).
# Later sessions: copies Drive cache -> local SSD (~3-5 min), then trains from
# local disk. Val transforms deterministic; val loader NEVER shuffled.

from datasets import load_dataset, load_from_disk
from timm.data import create_transform, Mixup
from timm.loss import SoftTargetCrossEntropy
from torchvision import transforms as T

HF_DATASET = "clane9/imagenet-100"   # 100-class ImageNet subset, train/val splits

def get_imagenet100(cfg):
    drive_cache = os.path.join(cfg["drive_root"], "imagenet100_hf")
    local_cache = os.path.join(cfg["data_local"], "imagenet100_hf")

    if not os.path.exists(drive_cache):
        print("First run: downloading ImageNet-100 from HuggingFace -> Drive (one-time)...")
        ds = load_dataset(HF_DATASET)
        ds.save_to_disk(drive_cache)

    if not os.path.exists(local_cache):
        print("Copying dataset Drive -> local SSD...")
        os.makedirs(cfg["data_local"], exist_ok=True)
        os.system(f"cp -r '{drive_cache}' '{local_cache}'")

    ds = load_from_disk(local_cache)
    print(f"train: {len(ds['train'])}  val: {len(ds['validation'])}")
    return ds

# --- transforms: DeiT-style train aug, standard center-crop eval ---
def build_transforms(cfg):
    train_tf = create_transform(
        input_size=cfg["img_size"], is_training=True,
        auto_augment="rand-m9-mstd0.5-inc1",
        interpolation="bicubic",
        re_prob=0.25, re_mode="pixel", re_count=1,
        mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225),
    )
    val_tf = T.Compose([
        T.Resize(int(cfg["img_size"] / 0.9), interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(cfg["img_size"]),
        T.ToTensor(),
        T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])
    return train_tf, val_tf

class HFWrap(torch.utils.data.Dataset):
    def __init__(self, hf_split, tf):
        self.ds, self.tf = hf_split, tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        ex = self.ds[i]
        img = ex["image"].convert("RGB")
        return self.tf(img), ex["label"]

def build_loaders(cfg, ds, epoch):
    """Train loader is rebuilt every epoch with generator seeded by (seed + epoch):
    data order in epoch k is IDENTICAL whether or not the run was interrupted."""
    train_tf, val_tf = build_transforms(cfg)
    g = torch.Generator(); g.manual_seed(cfg["seed"] + epoch)
    train_loader = DataLoader(
        HFWrap(ds["train"], train_tf), batch_size=cfg["batch_size"],
        shuffle=True, generator=g, num_workers=cfg["num_workers"],
        pin_memory=True, drop_last=True, persistent_workers=False,
    )
    val_loader = DataLoader(
        HFWrap(ds["validation"], val_tf), batch_size=cfg["batch_size"],
        shuffle=False, num_workers=cfg["num_workers"], pin_memory=True,
    )
    return train_loader, val_loader

def build_mixup(cfg):
    return Mixup(
        mixup_alpha=cfg["mixup_alpha"], cutmix_alpha=cfg["cutmix_alpha"],
        label_smoothing=cfg["label_smooth"], num_classes=cfg["num_classes"],
    )


# ============================================================================
# CELL T3 — GFNet (vendored verbatim from raoyongming/GFNet, isotropic variant)
# ============================================================================
# Source: https://github.com/raoyongming/GFNet/blob/master/gfnet.py
# Trimmed to what we use (no hierarchical/DownLayer variants). Logic unchanged.

from timm.models.layers import DropPath, to_2tuple, trunc_normal_

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x); x = self.act(x); x = self.drop(x)
        x = self.fc2(x); x = self.drop(x)
        return x

class GlobalFilter(nn.Module):
    def __init__(self, dim, h=14, w=8):
        super().__init__()
        self.complex_weight = nn.Parameter(torch.randn(h, w, dim, 2, dtype=torch.float32) * 0.02)
        self.w = w; self.h = h

    def forward(self, x, spatial_size=None):
        B, N, C = x.shape
        if spatial_size is None:
            a = b = int(math.sqrt(N))
        else:
            a, b = spatial_size
        x = x.view(B, a, b, C)
        x = x.to(torch.float32)
        x = torch.fft.rfft2(x, dim=(1, 2), norm='ortho')
        weight = torch.view_as_complex(self.complex_weight)
        x = x * weight
        x = torch.fft.irfft2(x, s=(a, b), dim=(1, 2), norm='ortho')
        x = x.reshape(B, N, C)
        return x

class GFBlock(nn.Module):
    def __init__(self, dim, mlp_ratio=4., drop=0., drop_path=0.,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm, h=14, w=8):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.filter = GlobalFilter(dim, h=h, w=w)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(dim, int(dim * mlp_ratio), act_layer=act_layer, drop=drop)

    def forward(self, x):
        x = x + self.drop_path(self.mlp(self.norm2(self.filter(self.norm1(x)))))
        return x

class GFPatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=256):
        super().__init__()
        img_size = to_2tuple(img_size); patch_size = to_2tuple(patch_size)
        self.num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0])
        self.img_size = img_size; self.patch_size = patch_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class GFNet(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=100,
                 embed_dim=256, depth=12, mlp_ratio=4., drop_rate=0., drop_path_rate=0.1,
                 norm_layer=None):
        super().__init__()
        norm_layer = norm_layer or partial(nn.LayerNorm, eps=1e-6)
        self.patch_embed = GFPatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)
        h = img_size // patch_size
        w = h // 2 + 1
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            GFBlock(embed_dim, mlp_ratio, drop_rate, dpr[i], norm_layer=norm_layer, h=h, w=w)
            for i in range(depth)])
        self.norm = norm_layer(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        trunc_normal_(self.pos_embed, std=.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x).mean(1)
        return self.head(x)


# ============================================================================
# CELL T4 — MODEL REGISTRY
# ============================================================================

def build_model(cfg):
    a = cfg["arch"]
    if a == "gfnet_ti":
        m = GFNet(img_size=cfg["img_size"], num_classes=cfg["num_classes"],
                  embed_dim=256, depth=12, drop_path_rate=cfg["drop_path"])
    elif a in ("prism2d", "prism2d_crelu", "prism2d_chanfft"):
        m = PRISM2D(
            img_size=cfg["img_size"], num_classes=cfg["num_classes"],
            d_model=256, depth=10, dropout=0.1,
            fft_mode="spatial_channel" if a == "prism2d_chanfft" else "spatial",
            activation="crelu" if a == "prism2d_crelu" else "modrelu",
        )
    else:
        raise ValueError(a)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{a}: {n/1e6:.2f}M params")
    return m


# ============================================================================
# CELL T4b — MODEL INSPECTION (run before training; sanity + paper table numbers)
# ============================================================================

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

def param_breakdown(model, arch_name=""):
    """Per-component split, mirroring the paper's Total / No-Embed / Core style."""
    M = 1e6
    total = count_params(model)
    rows = []

    if isinstance(model, GFNet):
        embed  = count_params(model.patch_embed) + model.pos_embed.numel()
        blocks = count_params(model.blocks)
        filters = sum(count_params(b.filter) for b in model.blocks)
        mlps    = sum(count_params(b.mlp) for b in model.blocks)
        head   = count_params(model.head) + count_params(model.norm)
        rows = [("patch_embed + pos_embed", embed),
                ("blocks (total)", blocks),
                ("  - global filters", filters),
                ("  - MLPs", mlps),
                ("norm + head", head)]
        core = blocks
    elif isinstance(model, PRISM2D):
        embed  = count_params(model.rose)
        blocks = count_params(model.layers)
        filt   = sum(count_params(l.neural_filter) for l in model.layers)
        glu    = sum(count_params(l.value_proj) + count_params(l.gate_proj)
                     + count_params(l.out_real) + count_params(l.out_imag)
                     for l in model.layers)
        bridge = count_params(model.bridge) + count_params(model.final_norm)
        head   = count_params(model.head)
        rows = [("DynamicRoSE2D (embed)", embed),
                ("GHC layers (total)", blocks),
                ("  - neural filters", filt),
                ("  - C-GLU + out proj", glu),
                ("bridge + final norm", bridge),
                ("head", head)]
        core = blocks
    else:
        rows = [(n, count_params(m)) for n, m in model.named_children()]
        core = total

    print(f"\n{'='*52}\n{arch_name or type(model).__name__}\n{'='*52}")
    for name, n in rows:
        print(f"  {name:<28} {n/M:>8.3f}M  ({100*n/total:.1f}%)")
    print(f"  {'-'*48}")
    print(f"  {'TOTAL':<28} {total/M:>8.3f}M")
    print(f"  {'CORE (mixing blocks)':<28} {core/M:>8.3f}M")
    return total, core

def inspect_before_training(cfg=None, print_full=False):
    cfg = cfg or make_run_cfg()
    model = build_model(cfg)
    param_breakdown(model, cfg["arch"])
    if print_full:
        print(model)          # full module tree (long!)
    else:
        # compact: one line per top-level child
        for name, m in model.named_children():
            print(f"  .{name}: {type(m).__name__}"
                  + (f" x{len(m)}" if isinstance(m, nn.ModuleList) else ""))
    return model

# --- compare all archs side by side before spending any units ---
def compare_all_archs():
    base = dict(TRAIN_CFG)
    results = {}
    for arch in ("gfnet_ti", "prism2d", "prism2d_crelu"):
        cfg = make_run_cfg(dict(base, arch=arch))
        results[arch] = param_breakdown(build_model(cfg), arch)
    print("\nSUMMARY (Total / Core, M):")
    for a, (t, c) in results.items():
        print(f"  {a:<16} {t/1e6:6.2f} / {c/1e6:6.2f}")

inspect_before_training(print_full=True)   # one model, full tree
compare_all_archs()                        # the comparison table


# ============================================================================
# CELL T5 — LR SCHEDULE (functional: computed from global step -> no scheduler
#            state to save; resume is exact by construction)
# ============================================================================

def lr_at(step, steps_per_epoch, cfg):
    warm = cfg["warmup_epochs"] * steps_per_epoch
    total = cfg["epochs"] * steps_per_epoch
    if step < warm:
        return cfg["lr"] * step / max(1, warm)
    t = (step - warm) / max(1, total - warm)
    return cfg["min_lr"] + 0.5 * (cfg["lr"] - cfg["min_lr"]) * (1 + math.cos(math.pi * t))

def set_lr(optimizer, lr):
    for g in optimizer.param_groups:
        g["lr"] = lr


# ============================================================================
# CELL T6 — CHECKPOINT I/O (last.pt every epoch, best.pt on improvement,
#            metrics.csv appended; full training state for exact resume)
# ============================================================================

def save_ckpt(path, model, opt, epoch, global_step, best_acc, cfg):
    torch.save({
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "epoch": epoch,                # last COMPLETED epoch
        "global_step": global_step,
        "best_acc": best_acc,
        "cfg": {k: v for k, v in cfg.items() if k != "amp_dtype"},
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }, path)

def try_resume(cfg, model, opt):
    path = os.path.join(cfg["run_dir"], "last.pt")
    if not os.path.exists(path):
        return 0, 0, 0.0
    ck = torch.load(path, map_location="cpu")
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["optimizer"])
    torch.set_rng_state(ck["torch_rng"])
    if ck["cuda_rng"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(ck["cuda_rng"])
    print(f"Resumed {cfg['run_name']} after epoch {ck['epoch']} (best {ck['best_acc']:.2f}%)")
    return ck["epoch"] + 1, ck["global_step"], ck["best_acc"]

def log_metrics(cfg, row):
    path = os.path.join(cfg["run_dir"], "metrics.csv")
    new = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new:
            w.writeheader()
        w.writerow(row)


# ============================================================================
# CELL T7 — TRAIN / EVAL LOOP
# ============================================================================

@torch.no_grad()
def evaluate(model, loader, cfg):
    model.eval()
    criterion = nn.CrossEntropyLoss()

    top1 = top5 = n = 0
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.autocast("cuda", dtype=cfg["amp_dtype"], enabled=DEVICE.type == "cuda"):
            logits = model(x)
            loss = criterion(logits.float(), y)

        bs = y.size(0)
        total_loss += loss.item() * bs

        _, pred5 = logits.topk(5, dim=-1)
        top1 += (pred5[:, 0] == y).sum().item()
        top5 += (pred5 == y[:, None]).any(-1).sum().item()
        n += bs

    return total_loss / n, 100 * top1 / n, 100 * top5 / n

def train(cfg=None):
    cfg = cfg or make_run_cfg()
    set_seed(cfg["seed"])
    ds = get_imagenet100(cfg)

    model = build_model(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                            weight_decay=cfg["weight_decay"], betas=(0.9, 0.999))
    criterion = SoftTargetCrossEntropy()
    mixup = build_mixup(cfg)

    start_epoch, global_step, best_acc = try_resume(cfg, model, opt)
    steps_per_epoch = len(ds["train"]) // cfg["batch_size"]

    for epoch in range(start_epoch, cfg["epochs"]):
        # epoch-seeded shuffle: identical data order with or without interruptions
        train_loader, val_loader = build_loaders(cfg, ds, epoch)
        model.train()
        t0, run_loss = time.time(), 0.0

        for it, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            x, y_soft = mixup(x, y)
            set_lr(opt, lr_at(global_step, steps_per_epoch, cfg))
            with torch.autocast("cuda", dtype=cfg["amp_dtype"], enabled=DEVICE.type == "cuda"):
                loss = criterion(model(x), y_soft)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            run_loss += loss.item(); global_step += 1
            if it % 50 == 0:
                print(f"  e{epoch} it{it}/{steps_per_epoch} "
                      f"loss {run_loss/(it+1):.3f} lr {opt.param_groups[0]['lr']:.2e}", end="\r")

        val_loss, acc1, acc5 = evaluate(model, val_loader, cfg)
        dt = time.time() - t0
        print(f"\nepoch {epoch}: train_loss {run_loss/steps_per_epoch:.3f} | "
              f"val_loss {val_loss:.3f} | top1 {acc1:.2f} top5 {acc5:.2f} | {dt/60:.1f} min")

        best_acc = max(best_acc, acc1)
        save_ckpt(os.path.join(cfg["run_dir"], "last.pt"),
                  model, opt, epoch, global_step, best_acc, cfg)
        if acc1 >= best_acc:
            save_ckpt(os.path.join(cfg["run_dir"], "best.pt"),
                      model, opt, epoch, global_step, best_acc, cfg)
        log_metrics(cfg, dict(epoch=epoch,
                      train_loss=round(run_loss/steps_per_epoch, 4),
                      val_loss=round(val_loss, 4),
                      top1=round(acc1, 2),
                      top5=round(acc5, 2),
                      lr=opt.param_groups[0]["lr"],
                      minutes=round(dt/60, 1)))

    print(f"DONE {cfg['run_name']} | best top1: {best_acc:.2f}%")
    return best_acc



In [ ]:

# ============================================================================
# CELL T8 — LAUNCH
# ============================================================================
# Run order:
#   1. TRAIN_CFG["arch"]="gfnet_ti"  -> train()   # CALIBRATION run
#   2. TRAIN_CFG["arch"]="prism2d"   -> train()
#   3. TRAIN_CFG["arch"]="prism2d_crelu" -> train()
# Re-running the cell after a Colab disconnect auto-resumes from last.pt.
TRAIN_CFG["arch"] = "prism2d_crelu"
train()

In [ ]:
import torch
ck = torch.load(TRAIN_CFG["drive_root"] + "/prism2d/last.pt", map_location="cpu")
print("wd =", ck["cfg"]["weight_decay"], "| epochs done =", ck["epoch"] + 1)

In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
import os
print("CPU cores:", os.cpu_count())
# during training, watch GPU util in another way: !nvidia-smi -l 5 in a terminal,
# or just trust the L4==A100 evidence — that's already conclusive.

In [ ]:
# ============================================================================
# CELL T9 — TRAINING LOOP PROFILER
# ============================================================================
# Answers two questions with numbers instead of vibes:
#   1. Where does step time go? (data wait vs GPU compute)
#   2. Which num_workers is best on THIS runtime?
#
# Usage:
#   profile_training(arch="gfnet_ti")                  # diagnose current config
#   sweep_workers(arch="gfnet_ti", workers=(4,8,12))   # find the best worker count
#
# Interpretation:
#   data_wait%  > 30%  -> CPU-bound: raise workers / pre-resize dataset (Cell T2b)
#   data_wait%  < 10%  -> GPU-bound: pipeline is healthy, nothing to fix
#   in between         -> pre-resize will still help, workers won't much

import time
import torch

def profile_training(arch=None, n_steps=60, warmup=10, num_workers=None, batch_size=None):
    cfg = make_run_cfg(dict(TRAIN_CFG,
                            arch=arch or TRAIN_CFG["arch"],
                            num_workers=num_workers or TRAIN_CFG["num_workers"],
                            batch_size=batch_size or TRAIN_CFG["batch_size"]))
    ds = get_imagenet100(cfg)
    train_loader, _ = build_loaders(cfg, ds, epoch=0)
    steps_per_epoch = len(ds["train"]) // cfg["batch_size"]

    model = build_model(cfg).to(DEVICE)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    mixup = build_mixup(cfg)
    criterion = SoftTargetCrossEntropy()

    data_t, compute_t = [], []
    it = iter(train_loader)
    t_fetch = time.perf_counter()
    for step in range(n_steps):
        # ---- data wait: time blocked on the loader ----
        x, y = next(it)
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        dt_data = t0 - t_fetch

        # ---- compute: forward + backward + step ----
        x, y_soft = mixup(x, y)
        with torch.autocast("cuda", dtype=cfg["amp_dtype"], enabled=DEVICE.type == "cuda"):
            loss = criterion(model(x), y_soft)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        torch.cuda.synchronize()
        t1 = time.perf_counter()

        if step >= warmup:                 # skip worker spin-up + cudnn autotune
            data_t.append(dt_data)
            compute_t.append(t1 - t0)
        t_fetch = t1

    d, c = sum(data_t) / len(data_t), sum(compute_t) / len(compute_t)
    step_s = d + c
    epoch_min = step_s * steps_per_epoch / 60
    mem_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

    print(f"\n--- {cfg['arch']} | bs {cfg['batch_size']} | workers {cfg['num_workers']} ---")
    print(f"  data wait : {d*1000:7.1f} ms/step  ({100*d/step_s:.0f}%)")
    print(f"  compute   : {c*1000:7.1f} ms/step  ({100*c/step_s:.0f}%)")
    print(f"  step      : {step_s*1000:7.1f} ms  ->  {epoch_min:.1f} min/epoch "
          f"({steps_per_epoch} steps)")
    print(f"  peak VRAM : {mem_gb:.1f} GB")
    verdict = ("CPU-BOUND -> raise workers / pre-resize (T2b)" if d/step_s > 0.30
               else "GPU-BOUND -> pipeline healthy" if d/step_s < 0.10
               else "mixed -> pre-resize (T2b) recommended")
    print(f"  verdict   : {verdict}")

    del model, opt
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return dict(arch=cfg["arch"], workers=cfg["num_workers"],
                data_ms=d*1000, compute_ms=c*1000, epoch_min=epoch_min, vram_gb=mem_gb)

def sweep_workers(arch=None, workers=(4, 8, 12), n_steps=50):
    rows = [profile_training(arch=arch, n_steps=n_steps, num_workers=w) for w in workers]
    print("\nSWEEP SUMMARY:")
    for r in rows:
        print(f"  workers={r['workers']:>2}  data {r['data_ms']:6.1f} ms  "
              f"compute {r['compute_ms']:6.1f} ms  ->  {r['epoch_min']:.1f} min/epoch")
    best = min(rows, key=lambda r: r["epoch_min"])
    print(f"  BEST: num_workers={best['workers']} ({best['epoch_min']:.1f} min/epoch)")
    return rows

profile_training(arch="gfnet_ti")
sweep_workers(arch="gfnet_ti", workers=(4, 8, 12))
profile_training(arch="prism2d")   # run before launching PRISM: VRAM + step-time check